# TSE - Tribunal Superior Eleitoral

Example notebook for the TSE scraper.

The TSE jurisprudence portal ([jurisprudencia.tse.jus.br](https://jurisprudencia.tse.jus.br/)) requires a CAPTCHA token, generated in the browser and validated server-side, for every search. The scraper issues plain HTTP requests and cannot produce that token, so **the jurisprudence search is not implemented**: `cjsg` raises `NotImplementedError`.

What the portal serves without a CAPTCHA is covered. The portal of the regional electoral courts ([jurisprudencia-tres.tse.jus.br](https://jurisprudencia-tres.tse.jus.br/)) runs on the same services, with the same CAPTCHA on the search, so the catalogs take `tribunal=` and the full-text download works with the code of a TRE decision too:

| Method | Description |
|--------|-------------|
| `baixar_inteiro_teor` | Downloads the full-text PDF of a decision of the TSE or of a TRE, given its `codigoDecisao` |
| `baixar_inteiros_teores` | Same, for a collection of codes: one row per code, with the path or the failure reason |
| `listar_classes` | Case classes of the search form |
| `listar_relatores` | Rapporteurs of the search form |
| `listar_eleicoes` | Election years of the search form |
| `listar_normas` | Legislation of the search form |

In [1]:
import juscraper as jus

tse = jus.scraper("tse")

## Catalogs

Each catalog is one request and comes back as a DataFrame, with the values exactly as the portal returns them.

In [2]:
classes = tse.listar_classes()
print(classes.shape)
classes.head()

(141, 2)


,sigla_classe,descricao_classe
0,RESPE,Recurso Especial Eleitoral
1,AI,Agravo de Instrumento
2,REspEl,RECURSO ESPECIAL ELEITORAL
3,AREspEl,Agravo em Recurso Especial Eleitoral
4,PA,Processo Administrativo


In [3]:
relatores = tse.listar_relatores()
print(relatores.shape)
relatores.head()

(236, 1)


,relator
0,Admar Gonzaga
1,Affonso Augusto Moreira Penna Júnior
2,Affonso Celso
3,Affonso Celso de Assis Figueiredo Júnior
4,Afrânio Antônio Da Costa


In [4]:
tse.listar_eleicoes()["eleicao"].tolist()

['2026',
 '2024',
 '2022',
 '2020',
 '2018',
 '2017',
 '2016',
 '2014',
 '2012',
 '2010',
 '2008',
 '2006',
 '2004',
 '2002',
 '2000',
 '1998',
 '1996',
 '1994',
 '1992',
 '1990',
 '1950']

In [5]:
normas = tse.listar_normas()
print(normas.shape)
normas[["sigla", "tipo", "numero", "ano"]].head()

(75, 5)


,sigla,tipo,numero,ano
0,EC117,EMENDA CONSTITUCIONAL,117,2022
1,ADCT,ATO DAS DISPOSIÇÕES CONSTITUCIONAIS TRANSITÓRIAS,1988,1988
2,CLT,DECRETO-LEI,5452,1943
3,CFD,CONSTITUIÇÃO FEDERAL,1988,1988
4,CADH,TRATADO,0,1969


### Regional electoral courts (TREs)

The `listar_*` methods take `tribunal`: `"tse"` (default), one TRE (`"tre-sp"`) or `"tres"`, the 27 TREs together.

In [6]:
classes_tre_sp = tse.listar_classes(tribunal="tre-sp")
print(classes_tre_sp.shape)
classes_tre_sp.head()

(128, 2)


,sigla_classe,descricao_classe
0,RE,RECURSO ELEITORAL
1,REl,RECURSO ELEITORAL
2,PC,Prestação de Contas
3,RCAND,Registro de Candidatura
4,REC,RECURSO CÍVEL


In [7]:
tse.listar_eleicoes(tribunal="tres")["eleicao"].tolist()

['2028',
 '2026',
 '2024',
 '2022',
 '2021',
 '2020',
 '2019',
 '2018',
 '2017',
 '2016',
 '2015',
 '2014',
 '2013',
 '2012',
 '2011',
 '2010',
 '2009',
 '2008',
 '2007',
 '2006',
 '2005',
 '2004',
 '2003',
 '2002',
 '2000',
 '1998',
 '1996',
 '1994',
 '1992',
 '1990']

## Full text of a decision

`baixar_inteiro_teor` takes the decision code (`codigoDecisao`), which is the last segment of the full-text link shown by the portal (`.../rest/download/pdf/<code>`), and returns the path of the saved PDF. The file goes to `<diretorio>/inteiro_teor/<code>.pdf`; without `diretorio`, to the scraper's `download_path`.

The PDF is saved as the court publishes it. Text extraction is left to the caller.

In [8]:
import tempfile
from pathlib import Path

pasta = tempfile.mkdtemp()
caminho = Path(tse.baixar_inteiro_teor(2972172, diretorio=pasta))

print(caminho.relative_to(pasta), caminho.stat().st_size, "bytes")
print(caminho.read_bytes()[:8])

inteiro_teor/2972172.pdf 38777 bytes
b'%PDF-1.7'


### Request spacing

The full-text service refuses requests that come too close together (it accepted one every 10 s and refused at 5 s, measured on 16 November 2026). The scraper therefore waits `sleep_time` seconds (default 10) between consecutive `baixar_inteiro_teor` calls, so the cell below takes about 10 s. Change it with `jus.scraper("tse", sleep_time=...)`. The wait is shared by every scraper instance in the same Python process, because the service does not tell them apart; separate processes do not see each other.

If the service still refuses, the call raises `LimiteDeRequisicoesError` and nothing is saved; wait and call again. A code that does not exist raises `InteiroTeorIndisponivelError`. Both live in `juscraper.courts.tse.download`.

In [9]:
from juscraper.courts.tse.download import InteiroTeorIndisponivelError

try:
    tse.baixar_inteiro_teor(1, diretorio=pasta)
except InteiroTeorIndisponivelError as erro:
    print(type(erro).__name__, "-", erro)

InteiroTeorIndisponivelError - Decisão 1: o TSE respondeu 'text/plain;charset=UTF-8' sem PDF ('Erro ao tentar recuperar PDF: Falha ao recuperar arquivo do SitDoc.'); nada foi gravado.


### Several decisions at once

`baixar_inteiros_teores` takes a collection of codes and returns one row per code: `caminho_inteiro_teor` with the path of the PDF and `motivo_falha` with the reason when the download failed (`nao_pdf` for a response without a PDF, such as a code that does not exist; `limite_de_requisicoes` when the service refused the request). A failure does not stop the batch, and all of them come in a single `UserWarning` at the end. The calls keep `sleep_time` between them, so the cell below takes about 30 s. Code `3419640` is a decision of the TRE-RS: the full-text service is the same for the TSE and the TREs.

In [10]:
import warnings

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    baixados = tse.baixar_inteiros_teores([3419640, 1, 3522031], diretorio=pasta)

print(avisos[0].message)
baixados.assign(caminho_inteiro_teor=baixados["caminho_inteiro_teor"].map(lambda c: c and str(Path(c).relative_to(pasta))))

TSEScraper.baixar_inteiros_teores: 1 download(s) de inteiro teor falharam. Falhas: 1: nao_pdf. As linhas correspondentes saem com o conteúdo None.


,codigo_decisao,caminho_inteiro_teor,motivo_falha
0,3419640,inteiro_teor/3419640.pdf,None
1,1,NaN,nao_pdf
2,3522031,inteiro_teor/3522031.pdf,None
